In [1]:
import os
import sys
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

try:
    pio.renderers.default = "vscode"
except Exception:
    pio.renderers.default = "notebook_connected"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, HTML, FileLink

from src.visualization.static_charts import (
    plot_top_categories_bar,
    plot_document_type_counts,
    plot_rating_distribution,
    plot_rating_by_category_boxplot,
    plot_popularity_vs_content_length_scatter,
    plot_average_rating_over_years,
    plot_numeric_correlation_heatmap,
    plot_news_dashboard_subplots,
    generate_all_static_charts,
)

from src.visualization.interactive_charts import (
    interactive_popularity_vs_content_length,
    interactive_top_categories_bar,
    interactive_records_over_years,
    interactive_rating_by_category_boxplot,
    interactive_news_multi_layout,
    generate_all_interactive_charts,
)

from src.visualization.chart_generator import generate_visualizations

DATA_PATH = Path("data/processed/cleaned/cleaned_data.csv")
STATIC_DIR = Path("outputs/visualizations/static")
INTERACTIVE_DIR = Path("outputs/visualizations/interactive")

STATIC_DIR.mkdir(parents=True, exist_ok=True)
INTERACTIVE_DIR.mkdir(parents=True, exist_ok=True)


def show_interactive_chart(path, height=700):
    """
    Display a saved Plotly HTML chart inside the notebook.

    This uses an absolute file:// URI instead of a Windows relative path,
    which avoids blank iframe outputs in VS Code/Jupyter.
    It also shows a clickable fallback link so the chart can be opened
    directly in a browser if the frontend blocks iframe rendering.
    """
    path = Path(path).resolve()

    if not path.exists():
        raise FileNotFoundError(f"Interactive chart file not found: {path}")

    file_uri = path.as_uri()

    display(HTML(f"""
    <iframe
        src="{file_uri}"
        width="100%"
        height="{height}"
        style="border:1px solid #cccccc; border-radius:8px; background:white;">
    </iframe>
    <p style="font-family:Arial; margin-top:8px;">
        <a href="{file_uri}" target="_blank">
            Open interactive chart in browser
        </a>
    </p>
    """))


def _prepare_interactive_df(df):
    """
    Prepare the cleaned news dataset for direct Plotly rendering in the notebook.
    This avoids iframe rendering issues in VS Code by displaying Plotly figures directly.
    """
    data = df.copy()

    for col, default in {
        "title": "Untitled",
        "category": "unknown",
        "document_type": "unknown",
        "language": "unknown",
        "source_name": "unknown",
        "url": "",
    }.items():
        if col in data.columns:
            data[col] = (
                data[col]
                .fillna(default)
                .astype(str)
                .str.strip()
                .replace("", default)
            )

    for col in [
        "rating_score",
        "popularity",
        "content_length",
        "title_length",
        "published_year",
        "vote_average",
        "vote_count",
        "year",
        "wins",
        "losses",
    ]:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    if "published_year" not in data.columns and "year" in data.columns:
        data["published_year"] = data["year"]

    return data


def _save_and_display_plotly(fig, stem, height=700):
    """
    Save a Plotly figure as a standalone HTML file and display the live Plotly
    object directly in the notebook. This is more reliable in VS Code than
    iframe-loading a local HTML file.
    """
    INTERACTIVE_DIR.mkdir(parents=True, exist_ok=True)
    path = INTERACTIVE_DIR / f"{stem}.html"

    fig.update_layout(height=height)
    fig.write_html(str(path), include_plotlyjs=True, full_html=True)

    display(fig)
    display(HTML(
        f'<p style="font-family:Arial; margin-top:8px;">'
        f'<b>Saved HTML:</b> <code>{path}</code><br>'
        f'<a href="{path.resolve().as_uri()}" target="_blank">Open interactive chart in browser</a>'
        f'</p>'
    ))

    return str(path)


def build_popularity_vs_content_length_fig(df):
    data = _prepare_interactive_df(df)
    plot_data = data.dropna(subset=["content_length", "popularity", "rating_score"]).copy()

    fig = px.scatter(
        plot_data,
        x="content_length",
        y="popularity",
        color="category",
        size="rating_score",
        hover_name="title",
        hover_data={
            "document_type": True,
            "source_name": True,
            "published_year": True,
            "rating_score": ":.2f",
            "language": True,
            "content_length": True,
            "popularity": ":.2f",
        },
        labels={
            "content_length": "Content Length",
            "popularity": "Popularity",
            "category": "Category",
            "rating_score": "Rating Score",
        },
        title="Popularity vs Content Length — Interactive News Explorer",
        template="plotly_white",
        color_discrete_sequence=px.colors.qualitative.Vivid,
    )

    fig.update_layout(legend_title="Category", font=dict(family="Inter", size=13))
    return fig


def build_top_categories_bar_fig(df, n=10):
    data = _prepare_interactive_df(df)

    counts = (
        data.groupby("category")
        .agg(
            record_count=("record_id", "count"),
            avg_rating=("rating_score", "mean"),
            avg_popularity=("popularity", "mean"),
            avg_content_length=("content_length", "mean"),
        )
        .reset_index()
        .sort_values("record_count", ascending=False)
        .head(n)
        .sort_values("record_count", ascending=True)
    )

    fig = px.bar(
        counts,
        x="record_count",
        y="category",
        orientation="h",
        color="avg_rating",
        hover_name="category",
        hover_data={
            "record_count": True,
            "avg_rating": ":.2f",
            "avg_popularity": ":.2f",
            "avg_content_length": ":.1f",
        },
        labels={
            "record_count": "Number of Records",
            "category": "Category",
            "avg_rating": "Average Rating",
        },
        title=f"Top {n} News Categories by Record Count",
        template="plotly_white",
        color_continuous_scale="Viridis",
    )

    fig.update_layout(font=dict(family="Inter", size=13))
    return fig


def build_records_over_years_fig(df):
    data = _prepare_interactive_df(df)
    plot_data = data.dropna(subset=["published_year"]).copy()

    if plot_data.empty:
        yearly = pd.DataFrame({
            "published_year": [0],
            "record_count": [0],
            "avg_rating": [0],
            "avg_popularity": [0],
        })
    else:
        plot_data["published_year"] = plot_data["published_year"].astype(int)
        yearly = (
            plot_data.groupby("published_year")
            .agg(
                record_count=("record_id", "count"),
                avg_rating=("rating_score", "mean"),
                avg_popularity=("popularity", "mean"),
            )
            .reset_index()
            .sort_values("published_year")
        )

    fig = px.line(
        yearly,
        x="published_year",
        y="record_count",
        markers=True,
        hover_data={
            "record_count": True,
            "avg_rating": ":.2f",
            "avg_popularity": ":.2f",
        },
        labels={
            "published_year": "Published Year",
            "record_count": "Number of Records",
            "avg_rating": "Average Rating",
            "avg_popularity": "Average Popularity",
        },
        title="News Records Over Time",
        template="plotly_white",
    )

    fig.update_traces(line_width=2.5, marker=dict(size=8))
    fig.update_layout(font=dict(family="Inter", size=13))
    return fig


def build_rating_by_category_boxplot_fig(df):
    data = _prepare_interactive_df(df)

    top_categories = (
        data["category"]
        .fillna("unknown")
        .astype(str)
        .value_counts()
        .head(8)
        .index
        .tolist()
    )

    plot_data = data[
        data["category"].isin(top_categories)
    ].dropna(subset=["category", "rating_score"]).copy()

    category_order = (
        plot_data.groupby("category")["rating_score"]
        .median()
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    fig = px.box(
        plot_data,
        x="category",
        y="rating_score",
        color="category",
        category_orders={"category": category_order},
        hover_name="title",
        hover_data={
            "document_type": True,
            "source_name": True,
            "published_year": True,
            "popularity": ":.2f",
            "content_length": True,
        },
        labels={
            "category": "Category",
            "rating_score": "Rating Score",
        },
        title="Rating Score Distribution by News Category",
        template="plotly_white",
        color_discrete_sequence=px.colors.qualitative.Vivid,
    )

    fig.update_layout(showlegend=False, font=dict(family="Inter", size=13))
    return fig


def build_interactive_news_dashboard_fig(df):
    data = _prepare_interactive_df(df)

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            "Top Categories by Record Count",
            "Rating Score Distribution",
            "Records by Document Type",
            "Popularity vs Content Length",
        ),
        vertical_spacing=0.16,
        horizontal_spacing=0.12,
    )

    category_counts = data["category"].value_counts().head(10).sort_values(ascending=True)

    fig.add_trace(
        go.Bar(
            x=category_counts.values,
            y=category_counts.index,
            orientation="h",
            marker_color="#1a6faf",
            name="Category Count",
            hovertemplate="Category: %{y}<br>Records: %{x}<extra></extra>",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Histogram(
            x=data["rating_score"].dropna(),
            nbinsx=25,
            marker_color="#2ca02c",
            name="Rating Distribution",
            hovertemplate="Rating: %{x}<br>Count: %{y}<extra></extra>",
        ),
        row=1,
        col=2,
    )

    doc_counts = data["document_type"].value_counts().head(10)

    fig.add_trace(
        go.Bar(
            x=doc_counts.index,
            y=doc_counts.values,
            marker_color="#ff7f0e",
            name="Document Type Count",
            hovertemplate="Document type: %{x}<br>Records: %{y}<extra></extra>",
        ),
        row=2,
        col=1,
    )

    scatter_data = data.dropna(subset=["content_length", "popularity", "rating_score"]).copy()

    fig.add_trace(
        go.Scatter(
            x=scatter_data["content_length"],
            y=scatter_data["popularity"],
            mode="markers",
            marker=dict(
                color=scatter_data["rating_score"],
                colorscale="Viridis",
                showscale=True,
                colorbar=dict(title="Rating", x=1.02, len=0.45, y=0.16),
                size=8,
                opacity=0.7,
            ),
            text=scatter_data["title"],
            customdata=scatter_data[
                ["category", "document_type", "published_year"]
            ].fillna("unknown"),
            name="News Records",
            hovertemplate=(
                "%{text}<br>"
                "Category: %{customdata[0]}<br>"
                "Document type: %{customdata[1]}<br>"
                "Year: %{customdata[2]}<br>"
                "Content length: %{x}<br>"
                "Popularity: %{y}<extra></extra>"
            ),
        ),
        row=2,
        col=2,
    )

    fig.update_layout(
        title_text="News Media Monitoring Interactive Dashboard",
        title_font=dict(size=18),
        template="plotly_white",
        width=1150,
        showlegend=False,
        font=dict(family="Inter", size=11),
    )

    fig.update_xaxes(title_text="Records", row=1, col=1)
    fig.update_xaxes(title_text="Rating Score", row=1, col=2)
    fig.update_xaxes(title_text="Document Type", row=2, col=1)
    fig.update_xaxes(title_text="Content Length", row=2, col=2)

    fig.update_yaxes(title_text="Category", row=1, col=1)
    fig.update_yaxes(title_text="Count", row=1, col=2)
    fig.update_yaxes(title_text="Records", row=2, col=1)
    fig.update_yaxes(title_text="Popularity", row=2, col=2)

    return fig


print("Project root:", PROJECT_ROOT)
print("Data exists:", DATA_PATH.exists())


Project root: c:\Users\38760\Desktop\Unstructured_Data\News_Media_Monitoring_Pipeline_UD
Data exists: True


## 2. Load dataset

In [2]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Dataset shape:", df.shape)
print(df.columns.tolist())
display(df[["record_id", "title", "category", "document_type", "published_year", "rating_score", "popularity", "content_length", "language"]].head())

Dataset shape: (1318, 35)
['record_id', 'source_name', 'title', 'description', 'url', 'source_path', 'fetched_at', 'version', 'document_type', 'file_name', 'page_number', 'extraction_timestamp', 'extraction_library', 'text', 'category', 'published_date', 'preview_text', 'name', 'year', 'wins', 'losses', 'raw_text', 'processed_text', 'content_text', 'language', 'published_year', 'content_length', 'title_length', 'rating_score', 'overview', 'genres', 'popularity', 'original_language', 'vote_average', 'vote_count']


,record_id,title,category,document_type,published_year,rating_score,popularity,content_length,language
0,52,Election Update,politics,excel,2026.0,4.0,34.0,15,unknown
1,53,AI Market Growth,business,excel,2026.0,7.0,28.0,16,unknown
2,54,Sports Highlights,sports,excel,2026.0,5.0,19.0,17,unknown
3,55,Climate Policy Shift,politics,excel,2026.0,3.0,23.0,20,unknown
4,56,Café Culture Trends,culture,excel,2026.0,6.0,14.0,19,unknown


## 3. Static Chart — Top 10 News Categories by Record Count

**Question and chart choice:** Which categories dominate the dataset? A horizontal bar chart is used because it compares discrete category counts and handles long labels cleanly.

**Tufte principle:** The chart avoids unnecessary decoration and uses simple encodings such as position, bar length, and color only where useful.

In [3]:
fig, paths = plot_top_categories_bar(df, output_dir=STATIC_DIR)
display(fig)
plt.close(fig)
print(paths)

<Figure size 1000x600 with 1 Axes>

{'png': 'outputs\\visualizations\\static\\top_categories_bar.png', 'pdf': 'outputs\\visualizations\\static\\top_categories_bar.pdf'}


## 4. Static Chart — Document Types in the Pipeline

**Question and chart choice:** What document/source types make up the pipeline? A horizontal bar chart makes frequency comparison direct and readable.

**Tufte principle:** The chart avoids unnecessary decoration and uses simple encodings such as position, bar length, and color only where useful.

In [4]:
fig, paths = plot_document_type_counts(df, output_dir=STATIC_DIR)
display(fig)
plt.close(fig)
print(paths)

<Figure size 1100x600 with 1 Axes>

{'png': 'outputs\\visualizations\\static\\document_type_counts.png', 'pdf': 'outputs\\visualizations\\static\\document_type_counts.pdf'}


## 5. Static Chart — Rating Score Distribution

**Question and chart choice:** How are rating scores distributed? A histogram is the correct chart for the shape of one numeric variable.

**Tufte principle:** The chart avoids unnecessary decoration and uses simple encodings such as position, bar length, and color only where useful.

In [5]:
fig, paths = plot_rating_distribution(df, output_dir=STATIC_DIR)
display(fig)
plt.close(fig)
print(paths)

<Figure size 900x600 with 1 Axes>

{'png': 'outputs\\visualizations\\static\\rating_score_distribution.png', 'pdf': 'outputs\\visualizations\\static\\rating_score_distribution.pdf'}


## 6. Static Chart — Rating Score by Category

**Question and chart choice:** Do categories differ in rating spread? A boxplot shows median, spread, and outliers across groups.

**Tufte principle:** The chart avoids unnecessary decoration and uses simple encodings such as position, bar length, and color only where useful.

In [6]:
fig, paths = plot_rating_by_category_boxplot(df, output_dir=STATIC_DIR)
display(fig)
plt.close(fig)
print(paths)

<Figure size 1200x600 with 1 Axes>

{'png': 'outputs\\visualizations\\static\\rating_by_category_boxplot.png', 'pdf': 'outputs\\visualizations\\static\\rating_by_category_boxplot.pdf'}


## 7. Static Chart — Popularity vs Content Length

**Question and chart choice:** Is longer content associated with popularity? A scatter plot is appropriate for two numeric variables.

**Tufte principle:** The chart avoids unnecessary decoration and uses simple encodings such as position, bar length, and color only where useful.

In [7]:
fig, paths = plot_popularity_vs_content_length_scatter(df, output_dir=STATIC_DIR)
display(fig)
plt.close(fig)
print(paths)

<Figure size 1000x700 with 1 Axes>

{'png': 'outputs\\visualizations\\static\\popularity_vs_content_length_scatter.png', 'pdf': 'outputs\\visualizations\\static\\popularity_vs_content_length_scatter.pdf'}


## 8. Static Chart — News Volume and Average Rating Over Time

**Question and chart choice:** How do yearly volume and average rating relate? A dual-axis bar/line chart compares count and rating on the same time axis.

**Tufte principle:** The chart avoids unnecessary decoration and uses simple encodings such as position, bar length, and color only where useful.

In [8]:
fig, paths = plot_average_rating_over_years(df, output_dir=STATIC_DIR)
display(fig)
plt.close(fig)
print(paths)

<Figure size 1100x600 with 2 Axes>

{'png': 'outputs\\visualizations\\static\\average_rating_over_years.png', 'pdf': 'outputs\\visualizations\\static\\average_rating_over_years.pdf'}


## 9. Static Chart — Numeric Correlation Heatmap

**Question and chart choice:** Which numeric fields are related? A heatmap is appropriate for pairwise correlations.

**Tufte principle:** The chart avoids unnecessary decoration and uses simple encodings such as position, bar length, and color only where useful.

In [9]:
fig, paths = plot_numeric_correlation_heatmap(df, output_dir=STATIC_DIR)
display(fig)
plt.close(fig)
print(paths)

<Figure size 1000x800 with 2 Axes>

{'png': 'outputs\\visualizations\\static\\numeric_correlation_heatmap.png', 'pdf': 'outputs\\visualizations\\static\\numeric_correlation_heatmap.pdf'}


## 10. Static Chart — Static Dashboard Subplots

**Question and chart choice:** What is the overall static summary? A 2x2 dashboard combines the most useful views into one figure.

**Tufte principle:** The chart avoids unnecessary decoration and uses simple encodings such as position, bar length, and color only where useful.

In [10]:
fig, paths = plot_news_dashboard_subplots(df, output_dir=STATIC_DIR)
display(fig)
plt.close(fig)
print(paths)

<Figure size 1600x1200 with 4 Axes>

{'png': 'outputs\\visualizations\\static\\news_dashboard_subplots.png', 'pdf': 'outputs\\visualizations\\static\\news_dashboard_subplots.pdf'}


## Generate all static charts automatically

This verifies that all 8 static charts are generated as both PNG and PDF files.

In [11]:
static_paths = generate_all_static_charts(df, output_dir=STATIC_DIR)
print("Static charts:", len(static_paths))
print("Static files:", sum(len(item) for item in static_paths))
display(pd.DataFrame(static_paths))

Static charts: 8
Static files: 16


,png,pdf
0,outputs\visualizations\static\top_categories_b...,outputs\visualizations\static\top_categories_b...
1,outputs\visualizations\static\document_type_co...,outputs\visualizations\static\document_type_co...
2,outputs\visualizations\static\rating_score_dis...,outputs\visualizations\static\rating_score_dis...
3,outputs\visualizations\static\rating_by_catego...,outputs\visualizations\static\rating_by_catego...
4,outputs\visualizations\static\popularity_vs_co...,outputs\visualizations\static\popularity_vs_co...
5,outputs\visualizations\static\average_rating_o...,outputs\visualizations\static\average_rating_o...
6,outputs\visualizations\static\numeric_correlat...,outputs\visualizations\static\numeric_correlat...
7,outputs\visualizations\static\news_dashboard_s...,outputs\visualizations\static\news_dashboard_s...


# Interactive Plotly Charts

Each Plotly chart is saved as a self-contained HTML file and includes hover tooltips with multiple fields.

## 12. Interactive Chart — Popularity vs Content Length

**Question and chart choice:** An interactive scatter plot lets the user hover over records to inspect title, document type, source, year, rating, popularity, and content length.

In [12]:
fig = build_popularity_vs_content_length_fig(df)
path = _save_and_display_plotly(
    fig,
    "popularity_vs_content_length_interactive",
    height=700,
)
print(path)


outputs\visualizations\interactive\popularity_vs_content_length_interactive.html


## 13. Interactive Chart — Top Categories Bar

**Question and chart choice:** An interactive bar chart compares category counts and provides hover details such as average rating, popularity, and content length.

In [13]:
fig = build_top_categories_bar_fig(df, n=10)
path = _save_and_display_plotly(
    fig,
    "top_categories_interactive_bar",
    height=650,
)
print(path)


outputs\visualizations\interactive\top_categories_interactive_bar.html


## 14. Interactive Chart — Records Over Years

**Question and chart choice:** A line chart is used for temporal data. The current dataset has limited valid year diversity, so it mainly shows year availability.

In [14]:
fig = build_records_over_years_fig(df)
path = _save_and_display_plotly(
    fig,
    "records_over_years_interactive_line",
    height=600,
)
print(path)


outputs\visualizations\interactive\records_over_years_interactive_line.html


## 15. Interactive Chart — Rating by Category Boxplot

**Question and chart choice:** An interactive boxplot compares distributions across groups while allowing record-level hover inspection.

In [15]:
fig = build_rating_by_category_boxplot_fig(df)
path = _save_and_display_plotly(
    fig,
    "rating_by_category_interactive_boxplot",
    height=650,
)
print(path)


outputs\visualizations\interactive\rating_by_category_interactive_boxplot.html


## 16. Interactive Chart — Interactive News Dashboard

**Question and chart choice:** A Plotly Graph Objects dashboard combines four linked views into one interactive HTML file.

In [16]:
fig = build_interactive_news_dashboard_fig(df)
path = _save_and_display_plotly(
    fig,
    "interactive_news_dashboard",
    height=760,
)
print(path)


outputs\visualizations\interactive\interactive_news_dashboard.html


## Generate all interactive charts automatically

This verifies that all 5 interactive charts are saved as HTML.

In [17]:
interactive_paths = generate_all_interactive_charts(df, output_dir=INTERACTIVE_DIR)
print("Interactive charts:", len(interactive_paths))
display(pd.DataFrame({"interactive_html_path": interactive_paths}))

Interactive charts: 5


,interactive_html_path
0,outputs\visualizations\interactive\popularity_...
1,outputs\visualizations\interactive\top_categor...
2,outputs\visualizations\interactive\records_ove...
3,outputs\visualizations\interactive\rating_by_c...
4,outputs\visualizations\interactive\interactive...


# Automated Generator and CLI

Lab 12 requires an orchestrator module and CLI entry point.

In [18]:
result = generate_visualizations(
    data_path=DATA_PATH,
    static_dir=STATIC_DIR,
    interactive_dir=INTERACTIVE_DIR,
)
print("Dataset shape:", result["dataset_shape"])
print("Static charts:", result["static_chart_count"])
print("Static files:", result["static_file_count"])
print("Interactive charts:", result["interactive_chart_count"])

Generating static charts...
Generating interactive charts...
Dataset shape: (1318, 35)
Static charts: 8
Static files: 16
Interactive charts: 5


In [19]:
completed = subprocess.run(
    [sys.executable, "scripts/generate_visualizations.py"],
    capture_output=True,
    text=True,
)
print("Return code:", completed.returncode)
print(completed.stdout)
if completed.stderr:
    print("STDERR:")
    print(completed.stderr)

Return code: 0
Generating static charts...
Generating interactive charts...

Visualization generation complete.
Dataset shape: (1318, 35)
Static charts: 8
Static files: 16
Interactive charts: 5

Static outputs:
  PNG: outputs\visualizations\static\top_categories_bar.png
  PDF: outputs\visualizations\static\top_categories_bar.pdf
  PNG: outputs\visualizations\static\document_type_counts.png
  PDF: outputs\visualizations\static\document_type_counts.pdf
  PNG: outputs\visualizations\static\rating_score_distribution.png
  PDF: outputs\visualizations\static\rating_score_distribution.pdf
  PNG: outputs\visualizations\static\rating_by_category_boxplot.png
  PDF: outputs\visualizations\static\rating_by_category_boxplot.pdf
  PNG: outputs\visualizations\static\popularity_vs_content_length_scatter.png
  PDF: outputs\visualizations\static\popularity_vs_content_length_scatter.pdf
  PNG: outputs\visualizations\static\average_rating_over_years.png
  PDF: outputs\visualizations\static\average_rating_

# Final Verification

In [20]:
png_files = sorted(STATIC_DIR.glob("*.png"))
pdf_files = sorted(STATIC_DIR.glob("*.pdf"))
html_files = sorted(INTERACTIVE_DIR.glob("*.html"))

verification_df = pd.DataFrame({
    "output_type": ["Static PNG", "Static PDF", "Interactive HTML"],
    "expected_count": [8, 8, 5],
    "actual_count": [len(png_files), len(pdf_files), len(html_files)],
})
verification_df["passed"] = verification_df["expected_count"] == verification_df["actual_count"]

display(verification_df)

print("PNG files:", len(png_files))
for p in png_files:
    print(" ", p)

print("\nPDF files:", len(pdf_files))
for p in pdf_files:
    print(" ", p)

print("\nHTML files:", len(html_files))
for p in html_files:
    print(" ", p)

,output_type,expected_count,actual_count,passed
0,Static PNG,8,8,True
1,Static PDF,8,8,True
2,Interactive HTML,5,5,True


PNG files: 8
  outputs\visualizations\static\average_rating_over_years.png
  outputs\visualizations\static\document_type_counts.png
  outputs\visualizations\static\news_dashboard_subplots.png
  outputs\visualizations\static\numeric_correlation_heatmap.png
  outputs\visualizations\static\popularity_vs_content_length_scatter.png
  outputs\visualizations\static\rating_by_category_boxplot.png
  outputs\visualizations\static\rating_score_distribution.png
  outputs\visualizations\static\top_categories_bar.png

PDF files: 8
  outputs\visualizations\static\average_rating_over_years.pdf
  outputs\visualizations\static\document_type_counts.pdf
  outputs\visualizations\static\news_dashboard_subplots.pdf
  outputs\visualizations\static\numeric_correlation_heatmap.pdf
  outputs\visualizations\static\popularity_vs_content_length_scatter.pdf
  outputs\visualizations\static\rating_by_category_boxplot.pdf
  outputs\visualizations\static\rating_score_distribution.pdf
  outputs\visualizations\static\top_

# Visualization Choices Documentation

This Lab 12 solution follows Tufte's graphical excellence principles: maximize data-ink ratio, avoid chartjunk, choose chart types that match the question, and make comparisons direct.

| Chart | Question | Why selected |
|---|---|---|
| Top categories bar | Which categories dominate? | Bars compare discrete category counts clearly. |
| Document type counts | Which source types dominate? | Bars are best for frequency comparison. |
| Rating distribution | What is the shape of rating scores? | Histogram shows one numeric distribution. |
| Rating by category boxplot | How do category distributions differ? | Boxplots compare medians, spread, and outliers. |
| Popularity vs content length | Are longer records more popular? | Scatter plots show relationships between two numeric variables. |
| Average rating over years | How do count and average rating vary by year? | A bar + line chart compares two metrics on a shared time axis. |
| Correlation heatmap | Which numeric features relate? | Heatmaps encode pairwise correlation strength compactly. |
| Static dashboard | What are the main dataset patterns? | Multi-panel layout summarizes several questions without overloading one chart. |
| Interactive scatter | Which records are outliers? | Hover and zoom reveal record-level metadata. |
| Interactive category bar | Which categories dominate and what are their averages? | Interactive bars combine count with hover summaries. |
| Interactive time line | How does volume vary by year? | Lines are standard for temporal trends. |
| Interactive boxplot | How do category rating distributions differ? | Interactive boxes show distributions and record-level hover details. |
| Interactive dashboard | What are the main patterns interactively? | Subplots provide multiple coordinated views in one HTML file. |

Dataset limitations should be documented: in this cleaned dataset, `category` often reflects source/document groups, and valid `published_year` values are concentrated mostly around 2026. These limitations affect time-series interpretation but do not invalidate the visualization workflow.

# Final Lab 12 Summary

The notebook is complete when it runs from top to bottom without errors and the final verification table shows:

- Static PNG: 8
- Static PDF: 8
- Interactive HTML: 5